<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Consume HTTP APIs responsibly and build idempotent automation with timeouts, retries, validation, and safe configuration.</p>
</div>

## Learning objectives

- Explain request methods, status codes, headers, and JSON bodies.
- Use timeouts and explicit error handling for every request.
- Validate response shape before using data.
- Design repeatable automation that is safe to rerun.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## HTTP is an unreliable boundary

A client sends a method, URL, headers, and optional body; the server returns a status, headers, and body. Network calls can be slow, fail, return partial responses, or succeed with an unexpected schema. Always set a timeout and treat transport, HTTP status, decoding, and schema as separate failure stages.


In [ ]:
from urllib.parse import urlencode

base_url = "https://api.example.com/courses"
query = urlencode({"track": "full-stack-ai", "limit": 20})
request_url = f"{base_url}?{query}"

expected_contract = {
    "method": "GET",
    "timeout_seconds": 10,
    "success_status": 200,
    "required_response_keys": {"items", "next_page"},
}
print(request_url)
print(expected_contract)


## A defensive request function

The `requests` library gives a clear API for HTTP work. `raise_for_status()` turns non-success status codes into exceptions; `.json()` may still fail or produce an unexpected shape. Reuse a `Session` for repeated calls and configure retries only for safe, transient operations.


In [ ]:
def fetch_json(session, url: str, *, timeout: float = 10.0) -> dict:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError("expected a JSON object")
    return payload


# Real usage with the third-party requests package:
# import requests
# with requests.Session() as session:
#     data = fetch_json(session, "https://api.example.com/courses")


## Idempotent automation

An idempotent job produces the same desired state when repeated. Use stable record keys, write temporary files before replacement, checkpoint progress, and distinguish a retryable failure from invalid input. Configuration belongs in environment variables or config files; secrets never belong in notebooks or logs.


In [ ]:
def upsert_by_id(existing: list[dict], incoming: list[dict]) -> list[dict]:
    records = {record["id"]: dict(record) for record in existing}
    for record in incoming:
        if "id" not in record:
            raise ValueError("every record needs an id")
        records[record["id"]] = dict(record)
    return [records[key] for key in sorted(records)]


current = [{"id": 1, "status": "new"}]
updates = [{"id": 1, "status": "ready"}, {"id": 2, "status": "new"}]
once = upsert_by_id(current, updates)
twice = upsert_by_id(once, updates)
assert once == twice
print(once)


## Worked example: paginated API contract with a fake client

Injecting the client makes pagination logic testable without a live service.


In [ ]:
class FakeClient:
    def __init__(self, pages):
        self.pages = pages

    def get_page(self, page: int) -> dict:
        return self.pages[page]


def collect_items(client) -> list[dict]:
    page = 1
    items = []
    while page is not None:
        payload = client.get_page(page)
        if not isinstance(payload.get("items"), list):
            raise ValueError("items must be a list")
        items.extend(payload["items"])
        page = payload.get("next_page")
    return items


client = FakeClient({
    1: {"items": [{"id": 1}], "next_page": 2},
    2: {"items": [{"id": 2}], "next_page": None},
})
print(collect_items(client))


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Describe the four failure stages of an API request.
2. Write a paginated collector that stops on `next_page=None`.
3. Validate that each item contains `id`, `name`, and `updated_at`.
4. Make a file-export job idempotent and explain its retry policy.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
def validate_item(item: dict) -> dict:
    required = {"id", "name", "updated_at"}
    if missing := required - item.keys():
        raise ValueError(f"missing keys: {sorted(missing)}")
    return {key: item[key] for key in sorted(required)}


def collect_valid_pages(client):
    page, accepted, errors = 1, [], []
    while page is not None:
        payload = client.get_page(page)
        for item in payload.get("items", []):
            try:
                accepted.append(validate_item(item))
            except ValueError as error:
                errors.append({"item": item, "error": str(error)})
        page = payload.get("next_page")
    return accepted, errors


print("Retry safe GET requests for transient timeouts and selected 5xx responses.")
print("Do not blindly retry invalid input or non-idempotent writes.")


## Knowledge check

**1. Why must every request have a timeout?**

::: {.callout-note collapse="true"}
### Answer
Otherwise it can wait indefinitely and block the job.
:::

**2. What does `raise_for_status` not validate?**

::: {.callout-note collapse="true"}
### Answer
JSON decoding and application schema.
:::

**3. What makes automation idempotent?**

::: {.callout-note collapse="true"}
### Answer
Repeating it converges on the same intended state.
:::


## Recap

- Treat the network as unreliable.
- Validate every boundary.
- Design jobs that can be resumed and rerun safely.


<div class="lesson-nav">
<a href="14-testing-typing-quality.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Testing, Type Hints, and Code Quality</a>
<a href="16-sql-and-databases.html">SQL and Databases with Python <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
